In [ ]:
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


Found existing installation: unsloth 2025.12.8
Uninstalling unsloth-2025.12.8:
  Successfully uninstalled unsloth-2025.12.8
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-vysz77em/unsloth_9343251df50d4552b106cabb8597c190
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-vysz77em/unsloth_9343251df50d4552b106cabb8597c190
  Resolved https://github.com/unslothai/unsloth.git to commit 2eb6b0d5f363a60ed3792ea1f04250537ac66939
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2025.12.8-py3-none-any.whl size=382385 sha256=9a6ff4b4ad2dd0c14ca22fd69a65c824539b51b28c521ddbdd6762845e08c100
  Stored in directory: /tmp/pip-ephem-wheel-cache-w2shg8lx/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_SAVE_PATH = "/content/drive/MyDrive/llama-3.1-8b-rag-finetuned"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

In [ ]:
max_seq_length = 2048  # Choose any! Unsloth auto-supports RoPE Scaling
dtype = None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True  # Use 4bit quantization to reduce memory usage


In [ ]:
# Load the dataset
dataset_url="neural-bridge/rag-dataset-1200"
dataset = load_dataset(dataset_url, split="train")

In [ ]:
# see a row of data
sample = dataset[0]
print(sample.keys())

dict_keys(['context', 'question', 'answer'])


In [ ]:
# Loading models
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Step 5: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,  # Supports any, but = 0 is optimized
    bias = "none",  # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",  # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # Rank stabilized LoRA
    loftq_config = None,  # LoftQ
)

==((====))==  Unsloth 2025.12.8: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
def formatting_prompts_func(examples):
    contexts = examples["context"]
    queries = examples["question"]
    answers = examples["answer"]
    texts = []

    for context, query, answer in zip(contexts, queries, answers):
        # Create messages in chat format
        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant that answers questions based on the provided context."
            },
            {
                "role": "user",
                "content": f"Context: {context}\n\nQuestion: {query}"
            },
            {
                "role": "assistant",
                "content": answer
            }
        ]

        # Apply Llama 3.1 chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)

    return {"text": texts}

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
# Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,  # Can make training 5x faster for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,  # Adjust based on dataset size
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# Step 9: Train the model
trainer_stats = trainer.train()



Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/960 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 960 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: muhammadjafri456 (muhammadjafri456-wai-industries) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.023500
2,2.924900
3,2.586400
4,2.803400
5,2.495700
6,2.757000
7,2.310400
8,2.547900
9,2.384000
10,2.223600


train/epoch,▁▁▁▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/grad_norm,█▅▆▆▃▂▃▂▂▁▂▂▂▃▂▃▃▁▃▂▂▂▁▂▂▂▁▂▁▂▂▁▁▂▂▂▁▂▁▂
train/learning_rate,▁▂▄▅▇███▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▁
train/loss,██▆▇▅▄▆▅▄▄▄▅▅▄▄▅▅▄▄▃▃▅▅▆▁▄▄▅▆▅▆▆▄▃▄▄▅▃▄▅
total_flos,2.029702902418637e+16
train/epoch,0.5
train/global_step,60
train/grad_norm,0.3577
train/learning_rate,0.0
train/loss,2.3598


In [ ]:
model.save_pretrained(DRIVE_SAVE_PATH)
tokenizer.save_pretrained(DRIVE_SAVE_PATH)

('/content/drive/MyDrive/llama-3.1-8b-rag-finetuned/tokenizer_config.json',
 '/content/drive/MyDrive/llama-3.1-8b-rag-finetuned/special_tokens_map.json',
 '/content/drive/MyDrive/llama-3.1-8b-rag-finetuned/chat_template.jinja',
 '/content/drive/MyDrive/llama-3.1-8b-rag-finetuned/tokenizer.json')